This is phase-2 implementation where we will clean the data and make it appropriate for downstream users.

In [0]:
from pyspark.sql.functions import col, expr,year,month,day
input_df = spark.read.table('kafka.default.user_activity_data_gen_bronze')

#remove unnecesary columns
without_unnecessary_columns_df = input_df.select('data.*')

#convert time_utc from milliseconds to timestamp, then convert from UTC to IST
# IST is UTC + 5 hours 30 minutes = 19800 seconds
append_timezone_df = without_unnecessary_columns_df.withColumn(
    'timestamp', 
    expr('CAST((time_utc/1000 + 19800) AS timestamp)')
).drop('time_utc')

#drop duplicates
without_duplicates_df  = append_timezone_df.dropDuplicates(['user_id','action_id', 'timestamp'])


In [0]:
from pyspark.sql.functions import current_timestamp,lit
#load country_code_lookup table
country_df = spark.read.table('kafka.default.country_code_lookup')

#check for valid country codes
valid_records_df = without_duplicates_df.join(country_df,'country_code','inner').drop('country').drop(country_df.country_code)

#check for valid country codes
invalid_records_df = without_duplicates_df.join(country_df,'country_code','left_anti').drop('country_code')

#save records containing invalid country codes
invalid_records_df.withColumn('run_id',lit('123456')).\
    withColumn('rejected_timestamp',current_timestamp()).\
    withColumn('error_message',lit('invalid country code')).\
    write.mode('append').\
    saveAsTable('kafka.default.user_activity_data_gen_silver_rejected')

In [0]:
#add year, month, day columns
enriched_df = valid_records_df.withColumn('year',year('timestamp')).\
    withColumn('month',month('timestamp')).\
    withColumn('day',day('timestamp'))
#save it in partition based on date and in delta format 
enriched_df.write.mode('overwrite').option('mergeSchema', True).partitionBy('year','month','day').format('delta').saveAsTable('kafka.default.user_activity_data_gen_silver')

#enriched_df.createOrReplaceTempView('enriched')

In [0]:
%sql
select distinct country_code from kafka.default.user_activity_data_silver;
